# json库
## 文件的读取与打印
* 读取：```js.load()```
    *
* 打印：```js.dump()```
* ```with open("vectorData.json",'r',encoding='utf-8')```
* 这句中的r/w是读取/写入控制符
    * with关键字可以让python自动关闭文件
## 字符串的转换
* python->json ```js.dumps()```
* json->python ```js.loads()```

In [18]:
import json

import numpy as np
import pandas as pd

# 读取.json文件
with open("vectorData.json",'r',encoding='utf-8') as f:
    data = f.read()
    print(data)


[
    {
        "group_name": "2d_task_1",
        "vectors": [[1,3],[1,2],[2,4],[3,1],[4,3],[5,5],[6,2],[7,7],[8,6],[9,8],[10,9]],
        "ori_axis": [[1,0],[0,1]],
        "tasks": [
            { "type": "axis_angle" },
            { "type": "change_axis", "obj_axis": [[2,1],[1,2]] },
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "axis_angle"}
        ]
    },
    {
        "group_name": "2d_task_2",
        "vectors": [[1,1],[2,0],[3,5],[4,2],[5,7],[6,4],[7,9],[8,6],[9,1],[10,8],[11,3],[12,10]],
        "ori_axis": [[1,1],[1,-1]],
        "tasks": [
            { "type": "area" },
            { "type": "axis_projection" },
            { "type": "change_axis", "obj_axis": [[3,2],[2,-3]] },
            { "type": "axis_projection"},
            { "type": "change_axis", "obj_axis": [[1,0],[0,1]] },
            { "type": "area"},
            { "type": "axis_angle" }
        ]
    },
    {
        "group_name": "2d_task_3",
        "vec

### 接下来准备正式接收数据

In [19]:
    data = json.loads(data)

    # 摊平数据
    vector_data = pd.json_normalize(
        data,
        # 定义要展开的层级
        record_path=['tasks'],
        # 定义要保留的外层字段
        meta = ['group_name', 'vectors', 'ori_axis'],
        errors='ignore',
    )
    # print(vector_data.info())
    # print(vector_data['type'].head())

    # 由于task中的各种东西过于混乱，我仅仅保留了目标坐标轴向量
    vector_data_clean = vector_data.dropna()
    vector_data_clean = vector_data_clean.drop('type',axis=1).copy()
    # 这里删掉这一列是因为我在后面类的定义里可以直接提取维度数
    vector_data_clean = vector_data_clean.drop('group_name',axis=1).copy()
    print(vector_data_clean.info())
    print(vector_data_clean.head())

    # 下面定义进行四种运算的类
    class AxCaculator:
        def __init__(self, ori_axis, obj_axis, vectors):
            self.ori_axis = np.array(ori_axis, dtype=float)
            self.obj_axis = np.array(obj_axis, dtype=float)
            self.vectors = np.array(vectors, dtype=float)
            self.n_dim = self.ori_axis.shape[0]

        # 1. 将vector转换为标准坐标系下的vector
        def to_standard(self):
            # 修复：vectors (k x n) @ ori_axis (n x n) 才是正确的矩阵乘法顺序
            standard_vector = self.vectors @ self.ori_axis
            return standard_vector

        # 2. 检查新的坐标轴向量是否合法
        def check_validity(self):
            # 修复：由于浮点数精度，不能直接用 == 0，应该用 np.isclose 或判断绝对值极小
            if np.isclose(np.linalg.det(self.obj_axis), 0):
                return False
            return True

        # 3. 再将其转换为新坐标系下的向量
        def obj_vector(self):
            if self.check_validity():
                standard_vec = self.to_standard()
                # 修复：转到新基底，应该乘新基底矩阵的【逆矩阵】
                new_basis_inv = np.linalg.inv(self.obj_axis)
                result = standard_vec @ new_basis_inv
                return result.tolist() # 转为列表方便存入 DataFrame
            return None

        # 4. 坐标系投影
        def shade_vector(self):
            if self.check_validity():
                # 归一化 obj_axis
                standard_axis_obj = self.obj_axis / np.linalg.norm(self.obj_axis, axis=1, keepdims=True)
                # 修复：需要用目标轴的转置 .T 来进行投影点积
                result = self.to_standard() @ standard_axis_obj.T
                return result.tolist()
            return None

        # 5. 坐标系夹角 (返回角度值)
        def theta_axis(self):
            if self.check_validity():
                standard_vec = self.to_standard()

                # 计算模长
                vec_norms = np.linalg.norm(standard_vec, axis=1, keepdims=True)
                axis_norms = np.linalg.norm(self.obj_axis, axis=1, keepdims=True)

                # 核心修复：防止除以 0。如果模长为 0，归一化结果也设为 0
                # 我们使用 np.where 或加一个极小的数字 1e-10
                norm_vec = np.divide(standard_vec, vec_norms, out=np.zeros_like(standard_vec), where=vec_norms!=0)
                norm_axis = np.divide(self.obj_axis, axis_norms, out=np.zeros_like(self.obj_axis), where=axis_norms!=0)

                cos_theta = norm_vec @ norm_axis.T
                cos_theta_safe = np.clip(cos_theta, -1.0, 1.0)
                angles = np.arccos(cos_theta_safe)
                angles_degree = np.degrees(angles)
                return angles_degree.tolist()
            return None

        # 6. 坐标系围成的高维体积
        def axis_value(self):
            if self.check_validity():
                gram_matrix = self.obj_axis @ self.obj_axis.T
                volume = np.sqrt(np.linalg.det(gram_matrix))
                return volume
            return None

        # ===== 新增：一个统一执行所有计算并返回字典的方法 =====
        def calculate_all(self):
            # 如果坐标轴无效，直接返回空值
            if not self.check_validity():
                return {
                    'obj_vector_calc': None,
                    'shade_vector_calc': None,
                    'theta_axis_calc': None,
                    'axis_volume': None
                }

            return {
                'obj_vector_calc': self.obj_vector(),
                'shade_vector_calc': self.shade_vector(),
                'theta_axis_calc': self.theta_axis(),
                'axis_volume': self.axis_value()
            }

    # 数据最终的处理输出逻辑
    # 定义处理每一行的函数
    def process_row(row):
        # 将当前行的数据喂给你的计算器
        calculator = AxCaculator(
            ori_axis=row['ori_axis'],
            obj_axis=row['obj_axis'],
            vectors=row['vectors']
        )
        # 调用统一计算方法，返回 pd.Series 自动展开为多列
        return pd.Series(calculator.calculate_all())

    # 1. 使用 apply 批量运算，生成一个全是新结果的 DataFrame
    new_calc_df = vector_data_clean.apply(process_row, axis=1)

    # 2. 将原数据和新结果按列方向（axis=1）拼接在一起
    df_final = pd.concat([vector_data_clean, new_calc_df], axis=1)

    print("\n========== 计算后拼接完成的 DataFrame ==========")
    print(df_final.head(7))

    # 将 df_final 导出为 JSON 文件
    # orient='records': 每行是一个字典，整个文件是一个大列表
    # force_ascii=False: 确保如果数据中有特殊字符（或中文）能正常显示，而不是显示为 \uXXXX
    # indent=4: 让生成的 JSON 文件有缩进，方便肉眼查看
    # df_final.to_json('任务数据文件.json', orient='records', force_ascii=False)

    import re

    def save_compact_json(df, filename):
        # 1. 先将 DataFrame 转换为 字典列表
        data_dict = df.to_dict(orient='records')

        # 2. 使用标准 json.dumps 生成带缩进的字符串
        # ensure_ascii=False 保证中文或特殊字符不被转码
        json_str = json.dumps(data_dict, indent=4, ensure_ascii=False)

        # 3. 正则表达式魔法：找到被拆分的数组并将其合并为一行
        # 这个正则会匹配形如 [ \n 1, \n 2 ] 的结构并替换为 [1, 2]
        # 我们执行两次，第一次处理最内层 [1, 2]，第二次处理外层 [[1,2], [3,4]]
        for _ in range(2):
            json_str = re.sub(
                r'\[\s+([^\[\]]+?)\s+\]',
                lambda m: "[" + re.sub(r'\s+', ' ', m.group(1)).strip() + "]",
                json_str
            )

        # 4. 写入文件
        with open(filename, 'w', encoding='utf-8') as f_:
            f_.write(json_str)

    save_compact_json(df_final, '任务数据文件.json')
    print("导出成功！文件名为: 任务数据文件.json")


<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 1 to 85
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   obj_axis  21 non-null     object
 1   vectors   21 non-null     object
 2   ori_axis  21 non-null     object
dtypes: object(3)
memory usage: 672.0+ bytes
None
             obj_axis                                            vectors  \
1    [[2, 1], [1, 2]]  [[1, 3], [1, 2], [2, 4], [3, 1], [4, 3], [5, 5...   
7   [[3, 2], [2, -3]]  [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...   
9    [[1, 0], [0, 1]]  [[1, 1], [2, 0], [3, 5], [4, 2], [5, 7], [6, 4...   
13   [[1, 1], [1, 0]]  [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0...   
16   [[1, 0], [0, 1]]  [[0, 5], [1, 4], [2, 3], [3, 2], [4, 1], [5, 0...   

             ori_axis  
1    [[1, 0], [0, 1]]  
7   [[1, 1], [1, -1]]  
9   [[1, 1], [1, -1]]  
13   [[2, 3], [3, 2]]  
16   [[2, 3], [3, 2]]  

========== 计算后拼接完成的 DataFrame ==========
              obj_ax